# UTS Data Mining II — AUDIO PROCESSING & K-MEANS

Contoh memakai 3 rekaman bawaan `librosa.ex()` (unduh pertama otomatis), pemeriksaan metadata, waveform, spectrogram amplitudo dan dB, ekstraksi fitur **MFCC per potongan 1 detik**, sampling maksimal 100 segmen audio, K-Means, evaluasi, dan PCA.

**Petunjuk:** Buka di Google Colab → `Runtime` → `Run all`, atau jalankan dari atas secara berurutan. Setiap sel **kode** menghasilkan output. Semua angka hasil yang dibahas pada catatan setelah sel adalah panduan interpretasi, **bukan hasil yang diasumsikan sudah diperoleh**.

## Panduan awal — membuka berbagai file

| Jenis file | Perintah paling singkat | Keterangan |
|---|---|---|
| CSV | `pd.read_csv('file.csv')` | Tabel dengan pemisah koma |
| TSV | `pd.read_csv('file.tsv', sep='\t')` | Tabel dengan pemisah tab |
| Excel | `pd.read_excel('file.xlsx')` | Mungkin perlu `openpyxl` |
| JSON | `pd.read_json('file.json')` | Data JSON yang cocok untuk tabel |
| TXT | `open('file.txt', encoding='utf-8').read()` | Teks bebas; tidak harus melalui pandas |
| Gambar | `PIL.Image.open('foto.jpg')` atau `cv2.imread(...)` | **Bukan** dibuka langsung dengan pandas |
| Audio | `librosa.load('audio.wav')` atau `sf.read(...)` | **Bukan** dibuka langsung dengan pandas |

Pandas terutama dipakai untuk data berbentuk tabel dan untuk menampung fitur yang sudah diekstraksi dari teks/gambar/audio. Dataset latihan di notebook ini diakses langsung tanpa perlu upload manual (koneksi internet diperlukan untuk unduhan pertama).

## Contoh tabel pandas

In [ ]:
import pandas as pd
from IPython.display import display

# Contoh objek DataFrame; .head() untuk melihat baris pertama.
contoh = pd.DataFrame({'jenis': ['Text', 'Image', 'Audio'],
                       'fitur': ['TF-IDF', 'Pixel', 'MFCC']})
display(contoh)
print('Ukuran tabel:', contoh.shape)

**Interpretasi:** Output menunjukkan tiga jenis data beserta contoh fitur numeriknya. Untuk pemeriksaan tabel gunakan `.head()`, `.shape`, `.info()`, `.isna().sum()`.

**Keyword:** `pandas read_csv sep tab`, `pandas head shape info`, `read_excel google colab`.

## 1. Import library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from IPython.display import display, Audio
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
print('librosa = pemrosesan suara, soundfile = info file/WAV, sklearn = clustering.')
print('Versi librosa:', librosa.__version__)

**Interpretasi:** `librosa` adalah library utama; `librosa.display` submodul untuk visualisasi. `sf` adalah alias dari library **soundfile** yang membaca metadata audio. **Keyword:** `librosa display specshow`, `soundfile info channels samplerate subtype`, `from sklearn.preprocessing import StandardScaler`.

## 2. Unduh contoh audio dari katalog librosa

In [ ]:
# Unduhan pertama butuh internet; berikutnya tersimpan di cache Colab.
nama_audio = ['trumpet', 'brahms', 'nutcracker']
berkas_audio = [librosa.ex(nama) for nama in nama_audio]
print('Daftar audio bawaan:')
for nama, path in zip(nama_audio, berkas_audio):
    print(nama, '->', path)

**Interpretasi:** `librosa.ex("trumpet")` mengembalikan **path** audio contoh, bukan langsung array sinyal. Tiga sumber digunakan supaya potongan audio punya variasi. **Keyword:** `librosa ex trumpet brahms nutcracker`, `librosa util list_examples`, `librosa example recordings`.

## 3. Muat waveform dan dengarkan

In [ ]:
# sr=22050 artinya audio di-resample ke 22.050 sampel/detik dan mono (default).
y, sr = librosa.load(berkas_audio[0], sr=22050)
print('Shape waveform mono:', y.shape)
print('Sample rate:', sr, 'Hz')
print('Durasi:', round(len(y) / sr, 2), 'detik')
print('Amplitudo min/max:', round(float(y.min()), 3), round(float(y.max()), 3))
display(Audio(y, rate=sr))

**Interpretasi:** `y` adalah sinyal **1D** karena dimuat mono. `sr` = sampling rate. `Audio(y,rate=sr)` dapat diputar langsung di Colab. `librosa.load` default melakukan mono conversion; ini belum tentu sama dengan jumlah channel pada file aslinya. **Keyword:** `librosa.load sr mono audio shape`, `IPython display Audio`, `audio sampling rate duration`.

## 4. Metadata channel, bit depth, bitrate WAV

In [ ]:
# Menyimpan waveform hasil pemuatan menjadi WAV PCM 16-bit, untuk contoh metadata pasti.
sf.write('contoh_audio_pcm16.wav', y, sr, subtype='PCM_16')
info = sf.info('contoh_audio_pcm16.wav')
bit_depth = 16  # Sesuai subtype yang ditentukan saat menulis WAV.
bitrate_pcm = info.samplerate * info.channels * bit_depth
print('Format / subtype:', info.format, '/', info.subtype)
print('Channel:', info.channels, '(1=mono; 2=stereo)')
print('Sample rate:', info.samplerate, 'Hz')
print('Frames/samples per channel:', info.frames)
print('Durasi:', round(info.duration, 2), 'detik')
print('Bit depth:', bit_depth, 'bit/sample')
print('Bitrate PCM (perkiraan tanpa header):', bitrate_pcm, 'bit/s =', round(bitrate_pcm/1000, 1), 'kbps')

**Interpretasi:** **Sample rate** = sampel per detik; **bit depth** = bit tiap sampel; **channel** = mono/stereo; **bitrate PCM** = `sample_rate × bit_depth × channel`. Ini berlaku untuk **WAV PCM yang baru dibuat**, bukan otomatis bitrate pada file MP3/OGG asli (yang terkompresi). **Keyword:** `soundfile info subtype PCM_16`, `audio bitrate sample rate bit depth channels formula`, `mono stereo audio metadata`.

## 5. Visualisasi waveform amplitudo terhadap waktu

In [ ]:
plt.figure(figsize=(12, 3))
librosa.display.waveshow(y, sr=sr)
plt.title('Waveform — amplitudo terhadap waktu')
plt.xlabel('Waktu (detik)')
plt.ylabel('Amplitudo (skala sinyal)')
plt.tight_layout()
plt.show()

**Interpretasi:** Waveform memperlihatkan variasi **amplitudo terhadap waktu**. Nilai amplitudo normalisasi bukan langsung nilai **dB**. **Keyword:** `librosa.display.waveshow waveform`, `audio waveform amplitude versus time`.

## 6. Spectrogram magnitudo/amplitudo

In [ ]:
# STFT mengubah sinyal waktu menjadi waktu × frekuensi.
S = np.abs(librosa.stft(y, n_fft=1024, hop_length=256))
print('Shape spektrogram magnitudo:', S.shape)
plt.figure(figsize=(11, 4))
librosa.display.specshow(S, sr=sr, hop_length=256, x_axis='time', y_axis='linear')
plt.colorbar(label='Magnitudo STFT (skala linear)')
plt.title('Spectrogram — magnitudo / amplitudo')
plt.tight_layout()
plt.show()

**Interpretasi:** Sumbu x = waktu, y = frekuensi dalam Hz, warna = **magnitudo komponen frekuensi**. STFT adalah *Short-Time Fourier Transform*. Frekuensi maksimum ≈ `sr/2` (Nyquist). **Keyword:** `librosa stft abs amplitude spectrogram`, `librosa.display.specshow x_axis time y_axis linear`, `n_fft hop_length explanation`.

## 7. Spectrogram skala desibel (dB)

In [ ]:
S_db = librosa.amplitude_to_db(S, ref=np.max)
print('Shape sebelum (magnitudo):', S.shape)
print('Shape sesudah (dB):', S_db.shape)
print('Rentang dB:', round(float(S_db.min()), 1), 'sampai', round(float(S_db.max()), 1))
plt.figure(figsize=(11, 4))
librosa.display.specshow(S_db, sr=sr, hop_length=256, x_axis='time', y_axis='linear')
plt.colorbar(label='dB relatif terhadap magnitudo maksimum')
plt.title('Spectrogram — skala dB')
plt.tight_layout()
plt.show()

**Interpretasi:** dB mengubah **skala tampilan** magnitudo menjadi logaritmik. `ref=np.max` membuat puncak relatif = **0 dB** dan komponen lebih lemah bernilai negatif. Ukuran matriks tidak berubah. **Keyword:** `librosa.amplitude_to_db ref np max`, `difference amplitude spectrogram and dB spectrogram`, `STFT spectrogram log scale`.

## 8. Ekstraksi MFCC dari potongan tiap sumber audio

In [ ]:
# Unit data clustering = potongan/segment berdurasi 1 detik; langkah antarsegmen 0,5 detik.
baris = []
for nama, path in zip(nama_audio, berkas_audio):
    sinyal, fs = librosa.load(path, sr=22050, duration=25)
    panjang = fs               # 1 detik
    langkah = fs // 2          # Geser 0,5 detik
    for awal in range(0, len(sinyal) - panjang + 1, langkah):
        potongan = sinyal[awal:awal + panjang]
        mfcc = librosa.feature.mfcc(y=potongan, sr=fs, n_mfcc=13)
        baris.append([nama, awal / fs] + mfcc.mean(axis=1).tolist())

kolom = ['sumber', 'waktu_mulai'] + [f'mfcc_{k}' for k in range(1, 14)]
df_audio_asli = pd.DataFrame(baris, columns=kolom)
print('Shape seluruh potongan audio:', df_audio_asli.shape)
display(df_audio_asli.head())

**Interpretasi:** **MFCC** adalah fitur numerik karakteristik spektral suara. Setiap potongan menghasilkan 13 fitur (rata-rata koefisien sepanjang frame waktu), ditambah `sumber` dan `waktu_mulai` sebagai **metadata**, bukan fitur model. Satu data clustering = satu **potongan audio**, bukan satu sampel waveform. **Keyword:** `librosa feature mfcc y sr n_mfcc`, `audio split into 1 second segments Python`, `MFCC mean axis 1`, `audio clustering features`.

## 9. Sampling 100 segmen dan standardisasi fitur

In [ ]:
df_audio = df_audio_asli.sample(n=min(100, len(df_audio_asli)), random_state=42).reset_index(drop=True)
X_audio = df_audio[[f'mfcc_{k}' for k in range(1, 14)]].to_numpy()
scaler = StandardScaler()
X_audio_scaled = scaler.fit_transform(X_audio)
print('Jumlah data setelah sampling:', len(df_audio))
print('Shape fitur sebelum standardisasi:', X_audio.shape)
print('Shape fitur setelah standardisasi:', X_audio_scaled.shape)
print('Distribusi sumber:')
print(df_audio['sumber'].value_counts())

**Interpretasi:** `.sample()` memilih maksimal 100 **segmen**. `StandardScaler` menyetarakan skala koefisien sebelum K-Means. Bentuk `(n_segmen,13)` tidak berubah saat standardisasi. **Keyword:** `sample dataframe 100 rows`, `StandardScaler fit_transform MFCC`, `why standardize before KMeans`.

## 10. Clustering K-Means dan evaluasi

In [ ]:
km_audio = KMeans(n_clusters=3, random_state=42, n_init=10)
df_audio['cluster'] = km_audio.fit_predict(X_audio_scaled)
print('Shape fitur input:', X_audio_scaled.shape)
print('Shape cluster output:', df_audio['cluster'].shape)
print('Jumlah tiap cluster:')
print(df_audio['cluster'].value_counts().sort_index())
print('Silhouette:', round(silhouette_score(X_audio_scaled, df_audio['cluster']), 3))
print('Crosstab sumber vs cluster:')
display(pd.crosstab(df_audio['sumber'], df_audio['cluster']))

**Interpretasi:** K-Means mengelompokkan **karakteristik MFCC**. Distribusi cluster antar sumber bisa membantu interpretasi, tetapi tidak berarti kita melakukan klasifikasi instrumen atau genre secara akurat. **Keyword:** `KMeans clustering MFCC audio`, `silhouette_score sklearn audio`, `pandas crosstab source cluster`.

## 11. Visualisasi cluster menggunakan PCA 2D

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_audio_2d = pca.fit_transform(X_audio_scaled)
print('Shape MFCC sebelumnya:', X_audio_scaled.shape)
print('Shape sesudah PCA:', X_audio_2d.shape)
plt.figure(figsize=(7, 5))
plt.scatter(X_audio_2d[:, 0], X_audio_2d[:, 1], c=df_audio['cluster'], cmap='viridis', s=24)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('K-Means segmen audio (fitur MFCC)')
plt.colorbar(label='Cluster')
plt.tight_layout()
plt.show()

**Interpretasi:** PCA mengubah 13 koefisien MFCC menjadi 2 dimensi **hanya untuk visualisasi**. K-Means berjalan pada 13 fitur terstandarisasi. **Keyword:** `PCA MFCC audio KMeans scatter`, `audio clustering visualize PCA 2 components`.

## 12. Ringkasan metadata dan alur audio

In [ ]:
print('METADATA WAV: channel =', info.channels, ', sample rate =', info.samplerate,
      ', bit depth =', bit_depth, ', bitrate PCM =', bitrate_pcm, 'bit/s')
print('SPECTROGRAM:', S.shape, '-> dB:', S_db.shape)
print('CLUSTERING:', df_audio_asli.shape, '-> sampling:', X_audio.shape,
      '-> scaling:', X_audio_scaled.shape, '-> PCA plot:', X_audio_2d.shape)
print('Selesai praktikum audio.')

**Interpretasi:** Alur utama: **baca file → cek metadata → waveform/spectrogram → potong audio → MFCC → sampling → scaling → K-Means → evaluasi → interpretasi**. Jangan menyamakan **bitrate**, **sample rate**, **bit depth**, **amplitudo**, dan **dB**. **Keyword:** `audio digital signal processing metadata sample rate bit depth`, `MFCC KMeans clustering workflow`.